# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [ ]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    op.payment_type,
    op.payment_installments,
    op.payment_sequential,
    oi.price,
    oi.freight_value,
    oi.product_id,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_payments op 
        ON o.order_id = op.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
    """)

In [ ]:
df_rfm_eda

In [ ]:
df_rfm_eda.describe()

#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [ ]:
df_rfm_eda.dtypes

In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'payment_type': 'category', 
              'payment_installments': 'int16',
              'payment_sequential': 'int16',
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

In [ ]:
df_rfm_eda.describe()

In [ ]:
df_rfm_eda.isna().sum()

In [ ]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

In [ ]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

In [ ]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

In [ ]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

In [ ]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


In [ ]:
test['sum_status']=test.sum(axis=1)

In [ ]:
test.loc[test['sum_status']!=1, :] 

####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [ ]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [ ]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

## Filterung nach der Bestellung mit den meisten Duplikaten

In [ ]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'
product_id = 'ebf9bc6cd600eadd681384e3116fda85'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[(df_rfm_eda['order_id'] == order_id) & (df_rfm_eda['product_id'] == product_id)]

order_data.head(63).sort_values('payment_sequential', ascending=True)

In [ ]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'
product_id = 'ebf9bc6cd600eadd681384e3116fda85'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[(df_rfm_eda['order_id'] == order_id) & (df_rfm_eda['product_id'] == product_id)]

order_data.head(63).sort_values('payment_sequential', ascending=True)

## Erkenntnis
Es handelt sich hier um keine Duplikate. 
Dadurch das die Payment Tabelle mit integriert wurde sieht es so als würden hier sehr viele Duplikate bestehen.
Allerding, hat er wie man oben sieht, ein Produkt 2 mal gekauft in einer Bestellung und diese in 21 Raten mit Gutschein bezahlt

Damit haben wir keine Duplikate im Datensatz

In [ ]:
df_payment = sql("""
SELECT *
FROM order_payments
    """)
test5 = df_payment[df_payment['payment_type'] == 'credit_card']
test5.sort_values(by='payment_sequential', ascending=False).head(10)

In [ ]:
test5[test5['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

In [ ]:
df_payment[df_payment['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

In [ ]:
df_payment[df_payment['order_id'] == 'b81ef226f3fe1789b1e8b2acac839d17']

In [ ]:
testvoucher = df_payment[df_payment['payment_type'] == 'voucher']
testvoucher.sort_values(by='payment_sequential', ascending=False).head(10)

In [ ]:
testvoucher[testvoucher['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c'].sort_values(by='payment_sequential', ascending=True)

In [ ]:
df_payment[df_payment['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c'].sort_values(by='payment_sequential', ascending=True)

In [ ]:
order_id = 'a079628ac8002126e75f86b0f87332e4'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)



In [ ]:
order_data.groupby('order_id')['payment_value'].nunique().sort_values(ascending=False)

In [ ]:
pd.crosstab(order_data['payment_value'], order_data['order_id'])

In [ ]:
normancode = sql("""
SELECT *
FROM order_payments
WHERE order_id = 'a079628ac8002126e75f86b0f87332e4'
ORDER BY payment_sequential;
                                  """)
normancode

In [ ]:
# NaNs nur in order_approved_at
df_rfm_eda[df_rfm_eda['order_approved_at'].isna()].head(10)


### EDA für zweite Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    r.review_score
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
JOIN order_items oi ON o.order_id = oi.order_id  
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
    """)

In [ ]:
df_pc_eda

In [ ]:
df_pc_eda.describe()

In [ ]:
df_pc_eda.dtypes

In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category', 
              'product_category_name_english': 'category',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

In [ ]:
df_pc_eda.describe()

In [ ]:
df_pc_eda.isna().sum()

In [ ]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

In [ ]:
test2 = pd.crosstab(df_pc_eda['order_status'], df_pc_eda['order_id']).T
test2.head(10)

In [ ]:
test2['sum_status']=test2.sum(axis=1)
test2.loc[test2['sum_status']!=1, :] 

In [ ]:
test2.loc[test2['sum_status']!=1, :].sort_values('sum_status', ascending=False)

### EDA für dritte Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [ ]:
df_service_eda

In [ ]:
df_service_eda.describe()


In [ ]:
df_service_eda.dtypes


In [ ]:
df_service_eda.isna().sum()